In [318]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt 
import seaborn as sns

In [319]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

# -----------------------------
# 1) Find the training dataset robustly
# -----------------------------
root = Path.cwd()

candidate_paths = [
    root / "training_dataset.csv",
    root / "models" / "training_dataset.csv",
    root.parent / "training_dataset.csv",
    root.parent / "models" / "training_dataset.csv",
]

data_path = None
for path in candidate_paths:
    if path.exists():
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(f"Could not find training_dataset.csv. Tried: {candidate_paths}")

print("Using data file:", data_path)

# -----------------------------
# 2) Load the current training data
# -----------------------------
df = pd.read_csv(data_path)

# -----------------------------
# 3) Drop leakage / non-feature columns
# -----------------------------
cols_to_drop = ["id", "total_seats", "booked_seats", "remaining_seats", "booking_date"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

# -----------------------------
# 4) Create X and y
# -----------------------------
X = df.drop(columns=["demand_ratio"])
y = df["demand_ratio"]

# -----------------------------
# 5) One-hot encode categorical columns
#    Keep binary flags and numeric columns as-is
#    Keep NaN values as true NaN (do NOT fill with 0)
# -----------------------------
categorical_cols = ["route", "flight_class"]
X = pd.get_dummies(X, columns=categorical_cols, dummy_na=False)

# -----------------------------
# 6) Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# -----------------------------
# 7) Train XGBoost model
# -----------------------------
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

# -----------------------------
# 8) Predict and evaluate
# -----------------------------
preds = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("RMSE:", round(rmse, 6))
print("R^2:", round(r2, 6))

# -----------------------------
# 9) Top 10 feature importances
# -----------------------------
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 10 feature importances:")
print(importances.head(10).to_string())

# -----------------------------
# 10) Save model only if result is in the healthy band
# -----------------------------
if 0.25 <= r2 <= 0.45:
    model_dir = None
    for base in [root, root.parent]:
        candidate_dir = base / "models"
        if candidate_dir.exists():
            model_dir = candidate_dir
            break

    if model_dir is None:
        model_dir = root

    model_path = model_dir / "demand_model.pkl"
    feature_cols_path = model_dir / "feature_columns.pkl"

    joblib.dump(model, model_path)
    joblib.dump(list(X.columns), feature_cols_path)

    print("\nModel saved successfully.")
    print("Model path:", model_path)
    print("Feature columns path:", feature_cols_path)
else:
    print("\nModel NOT saved because R^2 is outside the expected healthy range.")

Using data file: c:\Users\pc\Desktop\data\models\training_dataset.csv
RMSE: 0.147024
R^2: 0.46116

Top 10 feature importances:
current_price                0.286004
route_KHI-DXB                0.208235
base_fare                    0.075154
competitor_data_is_real      0.042173
price_vs_competitor_ratio    0.036032
competitor_min_price         0.032815
diesel_price                 0.028351
usd_to_pkr                   0.027879
petrol_price                 0.027482
is_holiday_window            0.026437

Model NOT saved because R^2 is outside the expected healthy range.


In [320]:
import joblib

joblib.dump(model, data_path.parent / "demand_model.pkl")
joblib.dump(list(X.columns), data_path.parent / "feature_columns.pkl")

print("Model saved successfully.")
print("Model path:", data_path.parent / "demand_model.pkl")

Model saved successfully.
Model path: c:\Users\pc\Desktop\data\models\demand_model.pkl
